# PixiJoy Inpaint (M1) — quality train (CPU OK)

**No GPU required.**

- `dist/` is **local Colab SSD** (fast). Checkpoints sync to Drive after train.
- `data/` may live on Drive (OK for reads).
- Train logs stream live (not buffered until exit).

Drive backup: `MyDrive/pixijoy_ml/inpaint/dist/`


In [ ]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print("TF", tf.__version__, "GPUs", gpus or "none (CPU OK)", flush=True)
print('accelerator=', 'GPU' if gpus else 'CPU', flush=True)


In [ ]:
import os, sys, urllib.request, zipfile, shutil
from pathlib import Path

IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    IN_COLAB = Path('/content').is_dir()

USE_DRIVE = IN_COLAB
ROOT = Path('/content/pixijoy_inpaint') if IN_COLAB else Path('.').resolve()
DRIVE = Path('/content/drive/MyDrive/pixijoy_ml/inpaint')
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)
print('cwd', ROOT, 'IN_COLAB', IN_COLAB, 'USE_DRIVE', USE_DRIVE, flush=True)

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    (DRIVE / 'dist').mkdir(parents=True, exist_ok=True)
    (DRIVE / 'data').mkdir(parents=True, exist_ok=True)

def _ensure_local_dir(name: str) -> Path:
    """Real local folder — NEVER a Drive symlink (FUSE hangs large .keras loads)."""
    p = Path(name)
    if p.is_symlink() or (p.exists() and p.resolve() != (ROOT / name).resolve() and 'drive' in str(p.resolve()).lower()):
        print(f'breaking Drive link for {name} -> {p.resolve()}', flush=True)
        if p.is_symlink():
            p.unlink()
        elif p.is_dir():
            shutil.rmtree(p)
        else:
            p.unlink()
    p.mkdir(parents=True, exist_ok=True)
    print(f'{name} is local SSD:', p.resolve(), flush=True)
    return p

def _link_data_to_drive() -> None:
    """Optional: data/ on Drive is OK (many small files). dist/ must stay local."""
    if not USE_DRIVE:
        Path('data').mkdir(parents=True, exist_ok=True)
        return
    local, target = Path('data'), DRIVE / 'data'
    target.mkdir(parents=True, exist_ok=True)
    if local.is_symlink():
        print('data -> Drive OK', flush=True)
        return
    if local.is_dir():
        for p in local.rglob('*'):
            if p.is_file():
                dest = target / p.relative_to(local)
                dest.parent.mkdir(parents=True, exist_ok=True)
                if not dest.is_file():
                    shutil.copy2(p, dest)
        shutil.rmtree(local)
    local.symlink_to(target)
    print(f'data -> {target}', flush=True)

ZIP_URL = 'https://raw.githubusercontent.com/Saikumar-S0906/Release-/main/inpaint_lab.zip'
LAB_ZIP_VERSION = '20260929b'
print(f'fetching lab zip v={LAB_ZIP_VERSION}…', flush=True)
ok = False
try:
    urllib.request.urlretrieve(ZIP_URL + f'?v={LAB_ZIP_VERSION}', 'inpaint_lab.zip')
    with zipfile.ZipFile('inpaint_lab.zip') as z:
        z.extractall('.')
    ok = Path('train.py').is_file()
    txt = Path('train.py').read_text(encoding='utf-8', errors='ignore')
    print('lab zip OK', ok, 'has early print=', 'before TensorFlow import' in txt, flush=True)
except Exception as e:
    print('zip fetch failed:', e, flush=True)
assert ok, 'lab zip missing train.py'
_ensure_local_dir('dist')
_link_data_to_drive()


In [ ]:
from pathlib import Path
import subprocess, sys

def run_stream(cmd):
    print('STREAM', ' '.join(cmd), flush=True)
    env = {**__import__('os').environ, 'PYTHONUNBUFFERED': '1'}
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    assert p.stdout is not None
    for line in p.stdout:
        print(line, end='', flush=True)
    rc = p.wait()
    print(f'[exit {rc}]', flush=True)
    return rc

print('prepare_quality_data…', flush=True)
rc = run_stream([sys.executable, '-u', 'prepare_quality_data.py', '--data', './data'])
assert rc == 0, f'prepare_quality_data failed rc={rc}'
assert (Path('data') / 'READY').is_file()
print('data READY ok', flush=True)


In [ ]:
from pathlib import Path
import hashlib, shutil, subprocess, sys, os
import tensorflow as tf

USE_DRIVE = globals().get('USE_DRIVE', False)
DRIVE = Path('/content/drive/MyDrive/pixijoy_ml/inpaint')
DRIVE_OUT = DRIVE / 'dist' / 'inpaint.keras'
OUT = Path('/content/pixijoy_inpaint/dist/inpaint.keras') if Path('/content').is_dir() else Path('dist') / 'inpaint.keras'
# Force absolute local path (never Drive)
if 'drive' in str(OUT.resolve()).lower():
    OUT = Path('/content/pixijoy_inpaint/dist/inpaint.keras')
OUT.parent.mkdir(parents=True, exist_ok=True)
print('OUT local?', OUT.resolve(), 'on_drive=', 'drive' in str(OUT.resolve()).lower(), flush=True)

if USE_DRIVE:
    DRIVE_OUT.parent.mkdir(parents=True, exist_ok=True)
    for name in (DRIVE_OUT.name, DRIVE_OUT.with_suffix('.ckpt.keras').name, 'train_state.json'):
        src, dst = DRIVE_OUT.parent / name, OUT.parent / name
        if src.is_file():
            print('copy Drive -> local SSD', src.name, f'{src.stat().st_size/1e6:.1f}MB', flush=True)
            shutil.copy2(src, dst)

ckpt = OUT.with_suffix('.ckpt.keras')
print('ckpt', ckpt.is_file(), ckpt, flush=True)

def run_stream(cmd):
    print('STREAM', ' '.join(map(str, cmd)), flush=True)
    env = {**os.environ, 'PYTHONUNBUFFERED': '1'}
    p = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    assert p.stdout is not None
    for line in p.stdout:
        print(line, end='', flush=True)
    return p.wait()

gpus = tf.config.list_physical_devices('GPU')
batch_flag = [] if gpus else ['--batch', '4']
cmd = [sys.executable, '-u', 'train.py', '--quality', '--out', str(OUT)] + batch_flag
if ckpt.is_file():
    cmd.append('--resume')
rc = run_stream(cmd)
assert rc == 0, f'train failed rc={rc}'

if USE_DRIVE:
    for p in OUT.parent.glob('*'):
        if p.is_file():
            shutil.copy2(p, DRIVE_OUT.parent / p.name)
            print('synced -> Drive', p.name, flush=True)

rc = run_stream([sys.executable, '-u', 'export_tflite.py', '--keras', str(OUT), '--out', str(OUT.parent / 'inpaint.tflite')])
assert rc == 0
if USE_DRIVE:
    shutil.copy2(OUT.parent / 'inpaint.tflite', DRIVE_OUT.parent / 'inpaint.tflite')
rc = run_stream([sys.executable, '-u', 'verify_contract.py', '--quality', '--soft-mae', '--data', './data', '--model', str(OUT.parent / '{artifact}')])
assert rc == 0, 'contract (shapes/dtype) failed'
p = OUT.parent / 'inpaint.tflite'
print('SHIP_OK', p.stat().st_size, hashlib.sha256(p.read_bytes()).hexdigest(), flush=True)


In [ ]:
from pathlib import Path
cands = [Path('dist/inpaint.tflite'), Path('/content/pixijoy_inpaint/dist/inpaint.tflite')]
p = next((c for c in cands if c.is_file()), None)
assert p, 'tflite missing'
print('artifact', p.resolve(), p.stat().st_size, flush=True)
try:
    from google.colab import files
    files.download(str(p))
except Exception:
    print('copy TFLite from', p)
